In [1]:
import pandas as pd
from pathlib import Path

In [2]:
db_path = Path(r"D:\BDDPablacomCoordinador")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\MedidasValorizadas")


In [3]:
lista_medidas = ["Medidas_Valorizadas_15min_Norte Distribución",
                    "Medidas_Valorizadas_15min_Norte",
                    "Medidas_Valorizadas_15min_Sur Distribución",
                    "Medidas_Valorizadas_15min_Sur"]

In [4]:
agg_rules = {
    'medida_3': ['sum', 'min'], 
    'CMg[CLP/KWh]': 'mean', 
    'valorizado_CLP': 'sum',
    'Fecha_Medicion': 'last',

    # --- Columnas Innecesarias ---
    #'Precio': innecesario, 
    #'nro_lt': 'innecesario',
    #'Cuarto de Hora': 'innecesario',
    #'descripcion': 'innecesario',
    #'error': 'innecesario',
    #'ID_Contrato': 'innecesario',
    #'Punto_Retiro': 'innecesario',
    #'Leido/Calculado': 'innecesario',
    #'Modo_Calculo': 'innecesario',
    #'medida_1': 'innecesario',
    #'medida_2': 'innecesario',
    #'medida_2a': 'innecesario',
    #'tipo': 'innecesario',
}

group_data = [
    'Hora', 
    'clave',
    'nombre_barra',
    'tension',
    'Zona',
    'Razon_Social',
    'RUT',
    'Nombre_Corto'
    ]

In [ ]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    origin, date, version = folder_date.name.split("_")
    
    if int(date) < 2505: # Procesamos solo los datos con el formato nuevo en este codigo
        continue

    # Base de datos antigua
    to_save_folder = to_save_path / f"{date}"
    
    if to_save_folder.is_dir():
        print(f"Los datos de {date} ya fueron procesados. Saltando...")
        continue  # Pasa a la siguiente iteración del for principal
        
    # Si la carpeta no existía, el script llega aquí y la crea para empezar a guardar
    to_save_folder.mkdir(parents=True, exist_ok=True)

    folder_medidas = folder_date / "02 Medidas por tipo"
    dfs = []
    dfs_problematics = []

    print(f"Empezando {date} ")

    for name_medida in lista_medidas:
        path_to_csv = folder_medidas / name_medida / "{}.csv".format(name_medida)

        df = pd.read_csv(path_to_csv, sep=";", dtype={"clave": str})  

        df = df[df["tipo"].str.contains("L")] # Solo clientes
        df = df[pd.to_numeric(df["RUT"].str.split("-").str[0].str.replace(".", "", regex=False), errors="coerce") >= 50000000] # Solo empresas 
        df["Hora"] = (df["Cuarto de Hora"] - 1) // 4 

        groups = df.groupby(by=group_data, sort=False) 
        groups_with_size = groups.size()
        good_groups_mask = (groups_with_size == 4)
        
        df_agregado = groups.agg(agg_rules)
        df_buenos = df_agregado[good_groups_mask].reset_index()
        df_buenos.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in df_buenos.columns]
        dfs.append(df_buenos)
        
        bad_groups_mask = ~good_groups_mask
        if bad_groups_mask.any(): # Si al menos un grupo falló
            print(f"{folder_date.name} tuvo problemas con {name_medida}")
            problematicos = groups.filter(lambda x: len(x) != 4)
            dfs_problematics.append(problematicos)
            
    if dfs:
        df_final_buenos = pd.concat(dfs, ignore_index=True)
        df_final_buenos.to_parquet(to_save_folder / f"{date}_medidas_horarias.parquet", engine="pyarrow", compression="snappy")
        
    if dfs_problematics:
        df_final_malos = pd.concat(dfs_problematics, ignore_index=True)
        df_final_malos.to_csv(to_save_folder / f"{date}_auditoria_errores_15min.csv", sep=";", index=False, encoding="utf-8")

Empezando 2505 


2603 con Medidas_Valorizadas_15min_Norte Distribución dio problemas en la columna index 2 entonces lo revisaremos manual, como conclusion a veces las claves son solo númericas y eso confunde al momento de cargar el csv. Entonces agregamos que la columna clave sea entendida como str
